<a href="https://colab.research.google.com/github/wrightalcode/Cartpole-Practice/blob/main/Cartpole_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ME 597 · In-class activity 2: Q-learning on CartPole

**Goal:** beat the hand-written policy from the lecture (about **42 steps**) using tabular Q-learning.

CartPole's state is four continuous numbers, but a Q-table needs a finite list of states. So before Q-learning can run, **you** decide how to cut each number into bins. That choice turns out to matter a lot.

**Your two tasks:**
1. Fill in the Q-learning update in Cell 3 (one line).
2. Try a few choices of `BINS` in Cell 2 (at least 3 sets), retrain, and write your best score on the board with the bins you used.

In [ ]:
# Cell 1: setup
%pip install -q gymnasium
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
env = gym.make("CartPole-v1")
print("State:  [cart position, cart velocity, pole angle, pole angular velocity]")
print("Actions: 0 = push left, 1 = push right")

In [ ]:
# Cell 2: choose how to discretize the state  <-- experiment here
# Number of bins for (x, x_dot, theta, theta_dot). A value of 1 means training will ignore that variable.

LIMITS = [(-2.4, 2.4), (-3.0, 3.0), (-0.21, 0.21), (-3.5, 3.5)]
EDGES = [np.linspace(lo, hi, n + 1)[1:-1] for (lo, hi), n in zip(LIMITS, BINS)]

def discretize(obs):
    """Map a continuous 4-D observation to a tuple of bin indices (a row of the Q-table)."""
    return tuple(int(np.digitize(v, e)) for v, e in zip(obs, EDGES))

print("Number of discrete states:", int(np.prod(BINS)))

In [ ]:
# Cell 3: tabular Q-learning
def train(episodes=2000, alpha=0.1, gamma=0.99, eps_decay=0.995, eps_min=0.05, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros(BINS + (2,))          # one row per discrete state, one column per action
    eps, returns = 1.0, []
    for ep in range(episodes):
        obs, _ = env.reset(seed=seed * 10_000 + ep)
        s, total = discretize(obs), 0.0
        while True:
            # epsilon-greedy action selection
            a = int(rng.integers(2)) if rng.random() < eps else int(np.argmax(Q[s]))
            obs, r, terminated, truncated, _ = env.step(a)
            s_next = discretize(obs)
            target = r if terminated else r + gamma * np.max(Q[s_next])
            Q[s + (a,)] += 0.0   # TODO: replace with the Q-learning update (see lecture slide)
            s, total = s_next, total + r
            if terminated or truncated:
                break
        eps = max(eps_min, eps * eps_decay)
        returns.append(total)
    return Q, np.array(returns)

Q, returns = train()
plt.plot(returns, alpha=0.3, label="episode return")
plt.plot(np.convolve(returns, np.ones(50) / 50, mode="valid"), label="50-episode average")
plt.axhline(42, color="k", ls="--", label="hand-written policy")
plt.xlabel("episode"); plt.ylabel("steps balanced"); plt.legend(); plt.show()

In [ ]:
# Cell 4: evaluate the greedy policy (no exploration) on 50 new episodes
scores = []
for ep in range(50):
    obs, _ = env.reset(seed=999_000 + ep)
    total = 0.0
    while True:
        obs, r, terminated, truncated, _ = env.step(int(np.argmax(Q[discretize(obs)])))
        total += r
        if terminated or truncated:
            break
    scores.append(total)
print(f"BINS = {BINS}  ->  {int(np.prod(BINS))} states  ->  greedy score: {np.mean(scores):.0f} steps (max 500)")

### Policy Visualization

In [ ]:
import imageio
from IPython.display import Image, display

# Initialize the environment with rendering enabled
env_render = gym.make("CartPole-v1", render_mode="rgb_array")
obs, _ = env_render.reset(seed=123)
frames = []

# Run a single episode
while True:
    frames.append(env_render.render())
    # Choose action greedily from Q-table
    action = int(np.argmax(Q[discretize(obs)]))
    obs, r, terminated, truncated, _ = env_render.step(action)
    if terminated or truncated:
        break

env_render.close()

# Save the frames as a GIF
gif_path = "trained_cartpole.gif"
imageio.mimsave(gif_path, frames, fps=30, loop=0)

# Display the GIF

**To consider:**
- What happens with `BINS = (1, 1, 1, 1)`? Why?
- Did more bins always help? What goes wrong when the table is very large?
- Run the same `BINS` with a different `seed` in `train(seed=...)`. How much does your score change?